# 06｜CNN 识别侦探项目 🔎

你将调查 CNN 为什么能识别手写数字、它容易把哪些数字认错，以及图片轻微移动后为什么可能失误。目标不是训练最大模型，而是用有限预算找到可信证据。

| 阶段 | 建议时间 | 必交证据 |
|---|---:|---|
| 热身与四徽章基线 | 20 分钟 | 基线诊断和徽章表 |
| 像素现场勘查 | 35 分钟 | 图片、矩阵与 Tensor 对应 |
| 卷积与池化取证 | 35 分钟 | 卷积响应和池化解释 |
| 三组单变量实验 | 55 分钟 | 公平对照记录 |
| 阅读并修改模型 | 45 分钟 | 一处有效修改、烟雾测试 |
| Boss 关：预算内改进 | 30 分钟 | 最多 3 次配置尝试 |
| 证据报告 | 20 分钟 | 300–500 字有边界结论 |
| **合计** | **240 分钟** | **约 4 小时** |

四小时用于观察、预测、改代码和写证据；全部训练仍是离线、小数据、CPU 单线程。

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

LAB_DIR = Path.cwd()
if not (LAB_DIR / "cnn_lab").is_dir():
    LAB_DIR = (LAB_DIR / "02_cnn_lab").resolve()
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from cnn_lab import (
    base_config, challenge_report, quick_demo, show_convolution,
    show_data, train_experiment
)

## 调查规则

- 固定使用 DEVICE = "cpu"；本项目不占课堂 GPU。
- 一组对照只改一个变量，并保持 seed、数据和 Epoch 一致。
- 全项目最多运行 8 次正式训练；图片观察不计入训练次数。
- 四枚徽章分别考查原图识别、右移一像素后的识别、参数量和更新次数。
- 不使用运行时间计分，也不靠增加 Epoch 刷分。

In [ ]:
DEVICE = "cpu"
baseline = quick_demo(
    channels=(8, 16),
    kernel_size=3,
    pooling="max",
    activation="relu",
    learning_rate=0.01,
    epochs=12,
    seed=42,
    device=DEVICE,
    show_features=True,
)
baseline_score = challenge_report(
    baseline,
    accuracy_target=0.96,
    shifted_target=0.60,
    parameter_budget=4_000,
    update_budget=300,
)

### 热身记录

基线得到 __ 枚徽章。最容易失去的是 __ 徽章。我认为原因是 __。

右移测试不是新的训练集，也不是证明模型普遍鲁棒；它只是一个可重复的小型压力测试。

## 第一案｜像素现场：一张数字怎样进入 CNN？（35 分钟）

先预测一张灰度图和一个 Batch 的 Tensor shape，再运行下方图片。选择一个容易混淆的数字，指出至少两处像素或笔画证据。

In [ ]:
gallery_figure, pixel_figure = show_data()

### 像素证词

单张图片 shape 是 __；64 张图片组成的 Batch shape 是 __；灰度通道位于第 __ 维。

我选择数字 __。它可能与数字 __ 混淆，因为 __。不要用“模型看懂了”代替具体像素或笔画描述。

## 第二案｜卷积核取证：哪些局部位置触发响应？（35 分钟）

运行前先预测：竖直边缘卷积核会在哪些笔画位置产生强响应？Max Pooling 与 Average Pooling 会保留相同信息吗？

In [ ]:
convolution_figure, pooling_figure = show_convolution()

### 卷积证词

卷积核通过 __ 在不同位置复用同一组权重。图中强响应位于 __。Pooling 以后空间尺寸从 __ 变为 __，而通道数 __。

## 第三案｜三组单变量实验（55 分钟）

下面分别减少通道、增大卷积核、替换池化。每次运行前先预测 Accuracy、右移测试、参数量和 Feature map shape 会怎样变化。

In [ ]:
TRIALS = {
    "减少通道": {"channels": (4, 8)},
    "增大卷积核": {"kernel_size": 5},
    "平均池化": {"pooling": "avg"},
}

trial_results = {}
for label, one_change in TRIALS.items():
    print(f"\n===== {label}：{one_change} =====")
    result = quick_demo(
        epochs=12, seed=42, device=DEVICE, show_features=False, **one_change
    )
    trial_results[label] = result
    challenge_report(
        result, accuracy_target=0.96, shifted_target=0.60,
        parameter_budget=4_000, update_budget=300
    )

### 对照记录

| 实验 | 我的预测 | 唯一修改项 | 原图 Accuracy | 右移 Accuracy | 参数量 | 图形证据 |
|---|---|---|---:|---:|---:|---|
| 减少通道 |  |  |  |  |  |  |
| 增大卷积核 |  |  |  |  |  |  |
| 平均池化 |  |  |  |  |  |  |

预测错误不扣分；没有写预测、同时改多个变量或只写“更好”才不构成证据。

## 安全门演示｜工作量来自分析，不来自占满服务器

下面故意提交 100 个 Epoch。程序应立即拒绝，不会开始训练。

In [ ]:
from dataclasses import replace

unsafe = replace(base_config(), epochs=100)
try:
    train_experiment(unsafe)
except ValueError as error:
    print("安全门已拦截：", error)

## 第四案｜读模型并增加一种激活函数（45 分钟）

1. 打开 cnn_lab/model.py，用自己的话解释 conv1、pool、conv2 和 classifier 的顺序。
2. 找到 _activation() 的 choices，在其中增加一种 PyTorch 已提供的激活函数，不新增依赖。
3. 把下面的名称换成你新增的名称并运行。
4. 执行 python test_cnn_smoke.py，保存通过截图。
5. 与 ReLU 比较原图、右移图、参数量和训练曲线。

这项任务只需修改激活映射，不应复制模型或训练循环。

In [ ]:
# 修改 model.py 前保持 tanh；完成后换成你新增的名称。
ACTIVATION_TO_TEST = "tanh"
activation_result = quick_demo(
    activation=ACTIVATION_TO_TEST,
    epochs=12,
    seed=42,
    device=DEVICE,
    show_features=False,
)

## Boss 关｜三次机会，争取四枚徽章（30 分钟）

目标：原图 Accuracy ≥ 97%，右移一像素 Accuracy ≥ 70%，参数量 ≤ 4000，更新次数 ≤ 300。最多尝试 3 个配置，每次先写预测。

可以修改 channels、kernel_size、pooling、activation、dropout、learning_rate；保持 epochs=12、seed=42。没有集齐徽章也可以完成，但要说明瓶颈和下一步。

In [ ]:
BOSS_CONFIG = dict(
    channels=(8, 16),
    kernel_size=3,
    pooling="max",
    activation="relu",
    dropout=0.0,
    learning_rate=0.01,
)

boss_result = quick_demo(
    epochs=12, seed=42, device=DEVICE, show_features=False, **BOSS_CONFIG
)
boss_score = challenge_report(
    boss_result, accuracy_target=0.97, shifted_target=0.70,
    parameter_budget=4_000, update_budget=300
)

## 最终提交｜300–500 字证据报告（20 分钟）

按以下顺序写：

1. 调查问题和运行前预测；
2. 三组实验中保持不变的条件与唯一修改项；
3. 至少两项数值证据，其中一项必须是右移 Accuracy 或参数量；
4. 一项具体图形证据：混淆数字、Feature map、Loss 或池化响应；
5. 代码修改位置，以及为什么不需要重写训练循环；
6. 使用“在当前 digits8 数据、seed 和训练预算下”限定结论。

提交：完成后的 Notebook、model.py 修改、烟雾测试截图、证据报告。

## 完成后：保存并释放资源

先保存 Notebook，再运行最后一格。继续实验时需要重新选择 Python 内核。

In [ ]:
%reset -f
import gc
import matplotlib.pyplot as plt
import torch
from IPython import get_ipython

plt.close("all")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("CNN 调查记录已完成，正在结束当前 Notebook 内核……")
get_ipython().kernel.do_shutdown(restart=False)